In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [28]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

26

In [30]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [31]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [32]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen3.8-27b", 
    api_key=os.environ["EXPLABS_API_KEY"],
    base_url="https://api.experientiallabs.ai/v1",
)

#### Chain - Context_generate | prompt | llm | strparser


In [33]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [34]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [35]:
rag_chain = get_context | prompt | llm

In [44]:
res = rag_chain.invoke("what is the name of patient and doctor ?")

In [45]:
print(res.content)

Patient name: **Ms. Nikita Chudhary**  
Doctor (referring doctor): **Dr. Nitin Nahar**
